In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, current_timestamp
from pyspark.sql import functions as F
import os

In [ ]:
try:
    spark.stop()
except:
    pass

In [ ]:
spark = (SparkSession.builder.appName("HealthcareDataProcessing_Location")
.config("spark.sql.files.ignoreCorruptFiles", "true")
.config("spark.driver.memory", "4g") 
.config("spark.executor.memory", "4g") 
.config("spark.memory.offHeap.enabled", "true") 
.config("spark.memory.offHeap.size", "2g") 
.config("spark.sql.session.timeZone", "UTC")
.master("local[*]")
.getOrCreate())

In [ ]:
silver_base_path = "../../data_lake/silver/silver_location/"
gold_base_path = "../../data_lake/gold/dim_location/"
gold_dimdate = "../../data_lake/gold/dim_date/"
gold_dimtime = "../../data_lake/gold/dim_time/"
gold_dimorganization = "../../data_lake/gold/dim_organization/"


In [ ]:
df_dimorganization = spark.read.format("parquet").load(gold_dimorganization)
df_dimdate = spark.read.format("parquet").load(gold_dimdate)
df_dimtime = spark.read.format("parquet").load(gold_dimtime)


In [ ]:
df_location = (spark.read.format("parquet").load(silver_base_path))

In [ ]:
df_location.show(5)

In [ ]:
df_inter = (
    df_location.alias("A").join(
        df_dimorganization.alias("B"),
        col("A.managing_organization_id") == col("B.organization_id"),
        "left"
    )
    .select(
        F.conv(F.substring(F.md5("A.location_id"), 1, 15), 16, 10).cast("bigint").alias("location_key"),
        col("A.location_id"),
        col("A.status"),
        col("A.name"),
        col("A.phone"),
        col("A.address_line"),
        col("A.city"),
        col("A.state"),
        col("A.postalCode"),
        col("A.country"),
        col("A.latitude"),
        col("A.longitude"),
        col("B.organization_key").alias("managing_organization_key"),
        current_timestamp().alias("gold_timestamp")
    )
)

In [ ]:
df_inter.write.mode("overwrite").format("parquet").save(gold_base_path)

In [ ]:
spark.stop()